In [88]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from itertools import combinations
from src.utils.helpers import clean_column_values
import json
import ast

import warnings
warnings.filterwarnings('ignore')

In [89]:
DATA_FOLDER = "data/"

movies_df = pd.read_csv(DATA_FOLDER + "v1_movies.csv")

for col in movies_df.columns:
    try:
        movies_df[col] = movies_df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    except:
        pass
    
movies_df

,wikipedia_id,freebase_id,title,languages,countries,genres,keywords,release_date,runtime,plot_summary,year_release_date,cold_war_side,character_western_bloc_representation,character_eastern_bloc_representation,western_bloc_values,eastern_bloc_values,theme
0,4213160.0,/m/0bq8q8,$,[English],[United States of America],"[Drama, Comedy, Action, Thriller, Heist, Crime...",NaN,1971-12-17,119.0,"Set in Hamburg, West Germany, several criminal...",1971,Western,"[Joe Collins, American bank security consultan...","[Sarge, corrupt U.S. Army sergeant, values rut...","[Cunning, heroism, cleverness, survival, Antih...","[Ruthlessness, violence, greed, betrayal, Anti...","[Heist, crime, betrayal, survival, tension]"
1,NaN,NaN,"$1,000 on the Black","[Deutsch, Italiano]","[Italy, Germany]",[Western],NaN,1966-12-18,104.0,Johnny Liston has just been released from pris...,1966,Western,"[Johnny Liston, justice, redemption, hero]","[Sartana, tyranny, betrayal, antagonist]","[Justice, redemption, individualism, personal ...","[Tyranny, fear, betrayal, oppression]","[Revenge, self-discovery, moral conflict, hero..."
2,NaN,NaN,"$10,000 Blood Money",NaN,[Soviet Union],"[Drama, Western]",NaN,1967-01-01,NaN,Hired by a Mexican landowner to rescue his dau...,1967,None,[None],[None],[None],[None],"[Betrayal, Greed, Bounty Hunter, Heist]"
3,NaN,NaN,"$100,000 for Ringo",[Italiano],[Italy],"[Drama, Western]","[spaghetti western, whipping]",1965-11-18,98.0,A stranger rides into Rainbow Valley where he'...,1965,None,[None],[None],[None],[None],"[Western, Frontier, Stranger, Rivalry, Money]"
4,2250713.0,/m/06z7m4,'68,[English],"[United States of America, Hungary]","[Drama, Coming of age, Family Drama, Period pi...",NaN,1988-01-01,98.0,The father escaped the Soviet invasion of Buda...,1988,None,[None],[None],[None],[None],"[Gay rights, counterculture, family conflict, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32434,NaN,NaN,Убить дракона,[Pусский],"[Russia, Germany]","[Fantasy, Drama]",NaN,1988-01-01,123.0,"Dragon is a bloody dictator, who kills every o...",1988,Eastern,"[Lancelot, heroism, selflessness, Knight arche...","[Dragon, oppression, fear, Tyrant archetype]","[Courage, freedom, hope, sacrifice]","[Oppression, fear, tyranny, control]","[Struggle against tyranny, fear, personal sacr..."
32435,NaN,NaN,’Round Midnight,"[English, Deutsch, Français]","[France, United States of America]",[Drama],"[jazz, musical, biography]",1986-09-23,133.0,Inside the Blue Note nightclub one night in 19...,1986,None,[None],[None],[None],[None],"[Jazz, friendship, creativity, Paris]"
32436,12680019.0,/m/02x07kx,…All the Marbles,[English],[United States of America],"[Drama, Comedy, Comedy-drama]",NaN,1981-01-01,113.0,Harry becomes manager of a tag team of gorgeou...,1981,None,[None],[None],[None],[None],"[Sports, Perseverance, Teamwork, Female Empowe..."
32437,NaN,NaN,…And the Fifth Horseman Is Fear,[Český],[Czech Republic],"[Drama, War]",NaN,1965-02-12,100.0,A Jewish doctor helps a political fugitive dur...,1965,Eastern,[None],"[Jewish doctor, compassion, sacrifice, hero]","[Humanity, empathy, moral duty, resistance]","[Oppression, fear, authoritarianism, survival]","[Oppression, resistance, moral dilemmas, WWII,..."


In [105]:
import re

global count

count = 0

def preprocess_side(row):
    # remove all non alphanumeric characters
    try:
        row['cold_war_side'] = re.sub(r'\W+', '', row['cold_war_side'])
    except:
        row['cold_war_side'] = "None"
    
    if row["character_western_bloc_representation"] is np.nan:
        row["character_western_bloc_representation"] = ['None']
    if row["character_eastern_bloc_representation"] is np.nan:
        row["character_eastern_bloc_representation"] = ['None']
    if row["western_bloc_values"] is np.nan:
        row["western_bloc_values"] = ['None']
    if row["eastern_bloc_values"] is np.nan:
        row["eastern_bloc_values"] = ['None']
    if row["theme"] is np.nan:
        row["theme"] = ['None']
        
    # print(row["western_bloc_values"][0])
    
    if (row["western_bloc_values"][0] == "None" and row["eastern_bloc_values"][0] == "None") and row['cold_war_side'] != "None":
        # print(row)
        # print("-----------------")
        global count
        count += 1
        row['cold_war_side'] = "None"
    
    return row

countries_representation = {
    'Soviet Union': 'Russia',
    'Soviet occupation zone': 'Russia',
    'Ukrainian SSR': 'Ukraine',
    'Ukranian SSR': 'Ukraine',
    'Uzbek SSR': 'Uzbekistan',
    'Georgian SSR': 'Georgia',
    'West Germany': 'Germany',
    'German Democratic Republic': 'Germany',
    'East Germany': 'Germany',
    'United Kingdom': 'United Kingdom',
    'England': 'United Kingdom',
    'Wales': 'United Kingdom',
    'Scotland': 'United Kingdom',
    'Northern Ireland': 'United Kingdom',
    'Socialist Federal Republic of Yugoslavia': 'Yugoslavia',
    'Federal Republic of Yugoslavia': 'Yugoslavia',
    'Republic of China': 'Taiwan',
    'South Korea': 'Korea',
    'North Korea': 'Korea',
    'Kingdom of Italy': 'Italy',
    'Republic of Macedonia': 'Macedonia',
    'Libyan Arab Jamahiriya': 'Libya',
    'Cote DIvoire': 'Côte d\'Ivoire',
    'Kingdom of Great Britain': 'United Kingdom',
    'Malayalam Language': 'India',
    'Syrian Arab Republic': 'Syria',
    'Kyrgyz Republic': 'Kyrgyzstan',
    'Slovak Republic': 'Czechoslovakia'
}

def preprocess_countries(row):
    # row['countries'] = clean_column_values(row['countries'])
    row['countries'] = set([countries_representation.get(string, string) for string in row['countries']]) if isinstance(row['countries'], list) else row['countries']
    # convert back to list
    row['countries'] = list(row['countries']) if isinstance(row['countries'], set) else row['countries']
    return row

movies_df2 = movies_df.copy()
movies_df2 = movies_df2.apply(preprocess_side, axis=1)
movies_df2 = movies_df2.apply(preprocess_countries, axis=1)

print(count, "false positives")

movies_df2

957 false positives


,wikipedia_id,freebase_id,title,languages,countries,genres,keywords,release_date,runtime,plot_summary,year_release_date,cold_war_side,character_western_bloc_representation,character_eastern_bloc_representation,western_bloc_values,eastern_bloc_values,theme
0,4213160.0,/m/0bq8q8,$,[English],[United States of America],"[Drama, Comedy, Action, Thriller, Heist, Crime...",NaN,1971-12-17,119.0,"Set in Hamburg, West Germany, several criminal...",1971,Western,"[Joe Collins, American bank security consultan...","[Sarge, corrupt U.S. Army sergeant, values rut...","[Cunning, heroism, cleverness, survival, Antih...","[Ruthlessness, violence, greed, betrayal, Anti...","[Heist, crime, betrayal, survival, tension]"
1,NaN,NaN,"$1,000 on the Black","[Deutsch, Italiano]","[Germany, Italy]",[Western],NaN,1966-12-18,104.0,Johnny Liston has just been released from pris...,1966,Western,"[Johnny Liston, justice, redemption, hero]","[Sartana, tyranny, betrayal, antagonist]","[Justice, redemption, individualism, personal ...","[Tyranny, fear, betrayal, oppression]","[Revenge, self-discovery, moral conflict, hero..."
2,NaN,NaN,"$10,000 Blood Money",NaN,[Russia],"[Drama, Western]",NaN,1967-01-01,NaN,Hired by a Mexican landowner to rescue his dau...,1967,None,[None],[None],[None],[None],"[Betrayal, Greed, Bounty Hunter, Heist]"
3,NaN,NaN,"$100,000 for Ringo",[Italiano],[Italy],"[Drama, Western]","[spaghetti western, whipping]",1965-11-18,98.0,A stranger rides into Rainbow Valley where he'...,1965,None,[None],[None],[None],[None],"[Western, Frontier, Stranger, Rivalry, Money]"
4,2250713.0,/m/06z7m4,'68,[English],"[United States of America, Hungary]","[Drama, Coming of age, Family Drama, Period pi...",NaN,1988-01-01,98.0,The father escaped the Soviet invasion of Buda...,1988,None,[None],[None],[None],[None],"[Gay rights, counterculture, family conflict, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32434,NaN,NaN,Убить дракона,[Pусский],"[Russia, Germany]","[Fantasy, Drama]",NaN,1988-01-01,123.0,"Dragon is a bloody dictator, who kills every o...",1988,Eastern,"[Lancelot, heroism, selflessness, Knight arche...","[Dragon, oppression, fear, Tyrant archetype]","[Courage, freedom, hope, sacrifice]","[Oppression, fear, tyranny, control]","[Struggle against tyranny, fear, personal sacr..."
32435,NaN,NaN,’Round Midnight,"[English, Deutsch, Français]","[United States of America, France]",[Drama],"[jazz, musical, biography]",1986-09-23,133.0,Inside the Blue Note nightclub one night in 19...,1986,None,[None],[None],[None],[None],"[Jazz, friendship, creativity, Paris]"
32436,12680019.0,/m/02x07kx,…All the Marbles,[English],[United States of America],"[Drama, Comedy, Comedy-drama]",NaN,1981-01-01,113.0,Harry becomes manager of a tag team of gorgeou...,1981,None,[None],[None],[None],[None],"[Sports, Perseverance, Teamwork, Female Empowe..."
32437,NaN,NaN,…And the Fifth Horseman Is Fear,[Český],[Czech Republic],"[Drama, War]",NaN,1965-02-12,100.0,A Jewish doctor helps a political fugitive dur...,1965,Eastern,[None],"[Jewish doctor, compassion, sacrifice, hero]","[Humanity, empathy, moral duty, resistance]","[Oppression, fear, authoritarianism, survival]","[Oppression, resistance, moral dilemmas, WWII,..."


In [106]:
movies_df2["cold_war_side"].value_counts()

cold_war_side
None       26029
Western     3656
Eastern     2754
Name: count, dtype: int64

In [107]:
# movies_df2["countires"] contains the countries that are involved in the movie. Create a set of all countries for all country in each lists of each film
all_countries = set()
for countries in movies_df2["countries"]:
    if isinstance(countries, list):
        all_countries.update(countries)

all_countries

{'Afghanistan',
 'Albania',
 'Algeria',
 'Angola',
 'Argentina',
 'Armenia',
 'Aruba',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bahamas',
 'Bahrain',
 'Bangladesh',
 'Belarus',
 'Belgium',
 'Bhutan',
 'Bolivia',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'Bulgaria',
 'Burkina Faso',
 'Burma',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Chile',
 'China',
 'Colombia',
 'Congo',
 'Costa Rica',
 "Cote D'Ivoire",
 'Croatia',
 'Cuba',
 'Czech Republic',
 'Czechoslovakia',
 'Democratic Republic of the Congo',
 'Denmark',
 'Ecuador',
 'Egypt',
 'Estonia',
 'Ethiopia',
 'Finland',
 'France',
 'Georgia',
 'Germany',
 'Ghana',
 'Greece',
 'Guinea',
 'Guinea-Bissau',
 'Hong Kong',
 'Hungary',
 'Iceland',
 'India',
 'Indonesia',
 'Iran',
 'Ireland',
 'Israel',
 'Italy',
 'Jamaica',
 'Japan',
 'Jordan',
 'Kazakhstan',
 'Korea',
 'Kuwait',
 'Kyrgyzstan',
 'Latvia',
 'Lebanon',
 'Libya',
 'Liechtenstein',
 'Lithuania',
 'Luxembourg',
 'Macedonia',
 'Malaysia',
 'Mali',
 'Martinique',
 'Mauritani

In [108]:
# remove uselss columns

movies_df3 = movies_df2.copy()
movies_df3 = movies_df3.drop(columns=["wikipedia_id", "freebase_id", "runtime"])

movies_df3

,title,languages,countries,genres,keywords,release_date,plot_summary,year_release_date,cold_war_side,character_western_bloc_representation,character_eastern_bloc_representation,western_bloc_values,eastern_bloc_values,theme
0,$,[English],[United States of America],"[Drama, Comedy, Action, Thriller, Heist, Crime...",NaN,1971-12-17,"Set in Hamburg, West Germany, several criminal...",1971,Western,"[Joe Collins, American bank security consultan...","[Sarge, corrupt U.S. Army sergeant, values rut...","[Cunning, heroism, cleverness, survival, Antih...","[Ruthlessness, violence, greed, betrayal, Anti...","[Heist, crime, betrayal, survival, tension]"
1,"$1,000 on the Black","[Deutsch, Italiano]","[Germany, Italy]",[Western],NaN,1966-12-18,Johnny Liston has just been released from pris...,1966,Western,"[Johnny Liston, justice, redemption, hero]","[Sartana, tyranny, betrayal, antagonist]","[Justice, redemption, individualism, personal ...","[Tyranny, fear, betrayal, oppression]","[Revenge, self-discovery, moral conflict, hero..."
2,"$10,000 Blood Money",NaN,[Russia],"[Drama, Western]",NaN,1967-01-01,Hired by a Mexican landowner to rescue his dau...,1967,None,[None],[None],[None],[None],"[Betrayal, Greed, Bounty Hunter, Heist]"
3,"$100,000 for Ringo",[Italiano],[Italy],"[Drama, Western]","[spaghetti western, whipping]",1965-11-18,A stranger rides into Rainbow Valley where he'...,1965,None,[None],[None],[None],[None],"[Western, Frontier, Stranger, Rivalry, Money]"
4,'68,[English],"[United States of America, Hungary]","[Drama, Coming of age, Family Drama, Period pi...",NaN,1988-01-01,The father escaped the Soviet invasion of Buda...,1988,None,[None],[None],[None],[None],"[Gay rights, counterculture, family conflict, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32434,Убить дракона,[Pусский],"[Russia, Germany]","[Fantasy, Drama]",NaN,1988-01-01,"Dragon is a bloody dictator, who kills every o...",1988,Eastern,"[Lancelot, heroism, selflessness, Knight arche...","[Dragon, oppression, fear, Tyrant archetype]","[Courage, freedom, hope, sacrifice]","[Oppression, fear, tyranny, control]","[Struggle against tyranny, fear, personal sacr..."
32435,’Round Midnight,"[English, Deutsch, Français]","[United States of America, France]",[Drama],"[jazz, musical, biography]",1986-09-23,Inside the Blue Note nightclub one night in 19...,1986,None,[None],[None],[None],[None],"[Jazz, friendship, creativity, Paris]"
32436,…All the Marbles,[English],[United States of America],"[Drama, Comedy, Comedy-drama]",NaN,1981-01-01,Harry becomes manager of a tag team of gorgeou...,1981,None,[None],[None],[None],[None],"[Sports, Perseverance, Teamwork, Female Empowe..."
32437,…And the Fifth Horseman Is Fear,[Český],[Czech Republic],"[Drama, War]",NaN,1965-02-12,A Jewish doctor helps a political fugitive dur...,1965,Eastern,[None],"[Jewish doctor, compassion, sacrifice, hero]","[Humanity, empathy, moral duty, resistance]","[Oppression, fear, authoritarianism, survival]","[Oppression, resistance, moral dilemmas, WWII,..."


In [110]:
# save to csv
movies_df4 = movies_df3.copy()
# put strings arround cold_war_side to avoid confusion with None

movies_df4["cold_war_side"] = movies_df4["cold_war_side"].apply(lambda x: f'"{x}"')

movies_df4.to_csv(DATA_FOLDER + "v1_movies_cleaned.csv", index=False)

movies_df4

,title,languages,countries,genres,keywords,release_date,plot_summary,year_release_date,cold_war_side,character_western_bloc_representation,character_eastern_bloc_representation,western_bloc_values,eastern_bloc_values,theme
0,$,[English],[United States of America],"[Drama, Comedy, Action, Thriller, Heist, Crime...",NaN,1971-12-17,"Set in Hamburg, West Germany, several criminal...",1971,"""Western""","[Joe Collins, American bank security consultan...","[Sarge, corrupt U.S. Army sergeant, values rut...","[Cunning, heroism, cleverness, survival, Antih...","[Ruthlessness, violence, greed, betrayal, Anti...","[Heist, crime, betrayal, survival, tension]"
1,"$1,000 on the Black","[Deutsch, Italiano]","[Germany, Italy]",[Western],NaN,1966-12-18,Johnny Liston has just been released from pris...,1966,"""Western""","[Johnny Liston, justice, redemption, hero]","[Sartana, tyranny, betrayal, antagonist]","[Justice, redemption, individualism, personal ...","[Tyranny, fear, betrayal, oppression]","[Revenge, self-discovery, moral conflict, hero..."
2,"$10,000 Blood Money",NaN,[Russia],"[Drama, Western]",NaN,1967-01-01,Hired by a Mexican landowner to rescue his dau...,1967,"""None""",[None],[None],[None],[None],"[Betrayal, Greed, Bounty Hunter, Heist]"
3,"$100,000 for Ringo",[Italiano],[Italy],"[Drama, Western]","[spaghetti western, whipping]",1965-11-18,A stranger rides into Rainbow Valley where he'...,1965,"""None""",[None],[None],[None],[None],"[Western, Frontier, Stranger, Rivalry, Money]"
4,'68,[English],"[United States of America, Hungary]","[Drama, Coming of age, Family Drama, Period pi...",NaN,1988-01-01,The father escaped the Soviet invasion of Buda...,1988,"""None""",[None],[None],[None],[None],"[Gay rights, counterculture, family conflict, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32434,Убить дракона,[Pусский],"[Russia, Germany]","[Fantasy, Drama]",NaN,1988-01-01,"Dragon is a bloody dictator, who kills every o...",1988,"""Eastern""","[Lancelot, heroism, selflessness, Knight arche...","[Dragon, oppression, fear, Tyrant archetype]","[Courage, freedom, hope, sacrifice]","[Oppression, fear, tyranny, control]","[Struggle against tyranny, fear, personal sacr..."
32435,’Round Midnight,"[English, Deutsch, Français]","[United States of America, France]",[Drama],"[jazz, musical, biography]",1986-09-23,Inside the Blue Note nightclub one night in 19...,1986,"""None""",[None],[None],[None],[None],"[Jazz, friendship, creativity, Paris]"
32436,…All the Marbles,[English],[United States of America],"[Drama, Comedy, Comedy-drama]",NaN,1981-01-01,Harry becomes manager of a tag team of gorgeou...,1981,"""None""",[None],[None],[None],[None],"[Sports, Perseverance, Teamwork, Female Empowe..."
32437,…And the Fifth Horseman Is Fear,[Český],[Czech Republic],"[Drama, War]",NaN,1965-02-12,A Jewish doctor helps a political fugitive dur...,1965,"""Eastern""",[None],"[Jewish doctor, compassion, sacrifice, hero]","[Humanity, empathy, moral duty, resistance]","[Oppression, fear, authoritarianism, survival]","[Oppression, resistance, moral dilemmas, WWII,..."
